# Multi-Hop Retrieval — Full System Evaluation (Kaggle)

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Your Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_dev.jsonl`
   - `musique_ans_v1.0_train.jsonl`
   - `model1_complement.pt`
   - `model2_scorer.pt`
4. The folder must be shared as **Anyone with the link can view**

**Compares:**
- MDR baseline (BM25 + dense)
- Graph + Cosine (no trained models)
- Graph + M1 + M2 — your full ComplementEncoder + ColBERT MaxSim system

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU (must be T4, not P100)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → T4 GPU")

cc = torch.cuda.get_device_properties(0).major
if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_60) — not supported by PyTorch 2.x.\n"
        "FIX: Settings → Accelerator → change P100 → T4 GPU"
    )

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

# Kaggle already has torch — only install missing packages
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'faiss-gpu' 'sentence-transformers>=2.2.0' gdown")
print("Dependencies installed")

In [ ]:
# 3. Download everything from Google Drive (data + model checkpoints)
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}")/1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up expected file paths
import os, shutil

drive = "/kaggle/working/drive_data"

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache", exist_ok=True)

# MuSiQue JSONL → symlink
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

# Model checkpoints → copy
for ckpt in ["model1_complement.pt", "model2_scorer.pt"]:
    src = f"{drive}/{ckpt}"
    dst = f"{WORK_DIR}/models/{ckpt}"
    if not os.path.exists(dst):
        print(f"  Copying {ckpt} ({os.path.getsize(src)/1e6:.0f} MB)...", flush=True)
        shutil.copy(src, dst)
    print(f"  {ckpt}: OK ({os.path.getsize(dst)/1e6:.0f} MB)")

print("\nAll paths ready")

---
## Run Full Evaluation
All 2,417 MuSiQue dev queries · 48,322 chunks · ~20 min on T4

In [ ]:
# 5. Run full system comparison
import os
os.chdir(WORK_DIR)

result_file = "/kaggle/working/eval_results.txt"
os.system(f"python run_full_system.py 2>&1 | tee {result_file}")
print("\nEvaluation complete")

In [ ]:
# 6. Display results table
with open("/kaggle/working/eval_results.txt") as f:
    log = f.read()

in_table = False
for line in log.split("\n"):
    if any(kw in line for kw in ["====", "System", "Recall", "MDR", "Graph", "FULL", "----", "R@"]):
        in_table = True
    if in_table:
        print(line)